# 🚗 Transformers × Carros
**Fine-tuning de um GPT-2 pequeno para o domínio automotivo**

Roteiro: dados → pré-processamento → modelo → treino → avaliação → comparação → demo.
Ative a GPU: *Ambiente de execução → Alterar tipo de ambiente → GPU T4*.

## 1. Preparação do ambiente

In [ ]:
import torch
print("GPU disponível:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
!pip -q install transformers datasets accelerate gradio requests matplotlib

### Enviar o projeto
Compacte a pasta `transformers-carros` em `.zip` e envie na célula abaixo
(ou clone seu repositório do GitHub no lugar). Se o notebook já estiver dentro da pasta do projeto, pule esta célula.

In [ ]:
import os, sys, zipfile
if not os.path.exists("src/config.py"):
    from google.colab import files
    enviado = files.upload()                      # selecione transformers-carros.zip
    nome = next(iter(enviado))
    zipfile.ZipFile(nome).extractall("/content")
    os.chdir("/content/transformers-carros")
sys.path.insert(0, os.getcwd())
print("Diretório de trabalho:", os.getcwd())

## 2. Coleta dos dados (Wikipédia)
Fonte definida em `src/config.py` (`FONTE_DADOS`). Se `data/artigos_carros.jsonl` já existir, ele é reaproveitado.

In [ ]:
from src import config
from src.fontes_dados import carregar_documentos
documentos = carregar_documentos()
print(documentos[0]["titulo"]); print(documentos[0]["texto"][:600])

## 3. Tokenizador e pré-processamento
Divisão treino/validação **por artigo** → tokenização → blocos de 256 tokens (`labels = input_ids`).

In [ ]:
from src.modelo import carregar_tokenizer, carregar_modelo, resumo_modelo
from src.preprocessamento import preparar_datasets
tok = carregar_tokenizer()
datasets = preparar_datasets(documentos, tok)
print(tok.decode(datasets["train"][0]["input_ids"][:60]))

## 4. Modelo base (GPT-2 pequeno em português)

In [ ]:
modelo = carregar_modelo(config.MODELO_BASE, tok)
resumo_modelo(modelo)

## 5. Fine-tuning
Previsão do próximo token (entropia cruzada), AdamW, fp16, early stopping.

In [ ]:
from src.treino import treinar
trainer = treinar(modelo, tok, datasets)

In [ ]:
from IPython.display import Image, display
display(Image(str(config.DIR_GRAFICOS / "curva_loss.png")))

## 6. Avaliação experimental
Perplexidade (domínio e fora do domínio), vocabulário automotivo, diversidade e repetição.

In [ ]:
from src.avaliacao import avaliar_tudo
base = carregar_modelo(config.MODELO_BASE, tok)           # cópia limpa, sem ajuste
ajustado = carregar_modelo(str(config.DIR_MODELO), tok)   # melhor checkpoint salvo
metricas = avaliar_tudo(base, ajustado, tok, datasets)
display(Image(str(config.DIR_GRAFICOS / "perplexidade.png")))

## 7. Comparação qualitativa: base × ajustado
Observe: vocabulário, alucinações (Ferrari, F1) e a piada do *Optimus Prime*.

In [ ]:
from src.geracao import comparar
resultados = comparar(base, ajustado, tok)

## 8. Demonstração ao vivo
Cria uma interface Gradio com link público. Se falhar, use a versão simples (próxima célula).

In [ ]:
from src.demo import criar_interface
app = criar_interface(base, ajustado, tok)
app.launch(share=True, debug=False)

In [ ]:
# Plano B (sem Gradio)
from src.geracao import gerar
prompt = "Optimus Prime é um caminhão"
print("BASE    :", gerar(base, tok, prompt, seed=42)[0])
print("AJUSTADO:", gerar(ajustado, tok, prompt, seed=42)[0])